In [ ]:
# ! pip install pandas numpy matplotlib pillow

In [ ]:
# Imports
import colorsys            # converts colors to lightness values, to make lighter/darker shades
import re                  # pattern matching on text, e.g. finding "8" in "week8"
from pathlib import Path   # handles file/folder paths the same on Windows and Mac/Linux

import matplotlib.pyplot as plt       # the plotting functions themselves
from matplotlib.colors import to_rgb  # turns "#2E6F4E" into numbers we can shade
import numpy as np                    # fast math on whole columns of numbers at once
import pandas as pd                   # tables: reading the CSV, grouping, counting

In [ ]:
# How many graphs to make. Only the TOP_N most common transitions get one,
# because a recording has thousands of different transitions and graphing all
# of them would take forever.
TOP_N = 10

# Which columns hold the cluster number and the stage name. The script tries
# these pairs in order and uses the first one the table has:
#   ("ClusterIdx", "Folder_Name")  -> Cluster_detail_results from MATLAB
#   ("Cluster", "Week_Number")     -> the visualization pipeline's combined_matrix
COLUMN_PAIRS = [("ClusterIdx", "Folder_Name"), ("Cluster", "Week_Number")]

# "Side" stages are treatment sessions such as L-DOPA or saline. They are drawn
# as diamonds after the main timeline and left out of the ranking. A stage
# counts as a side stage if its name contains one of these (any case) -- the
# same convention Dylan used. Numbered stages (like Week_Number 25) have no
# name to search, so list those with special_stages={25: "L-Dopa"}.
SEPARATE_KEYWORDS = ("ldop", "salin")

# Line colors: Dylan's lab colors (UTD_GREEN / UTD_ORANGE in his utils.py):
#     MitoPark  -> medium-dark green     #2E6F4E
#     controls  -> medium-bright orange  #F08C1E
# Several mice of one group each get a lighter/darker shade of it.
UTD_GREEN = "#2E6F4E"
UTD_ORANGE = "#F08C1E"

# Files that are neither MitoPark nor control get these instead, in order:
# blue, purple, red, brown, pink, cyan, olive, gray (never green or orange).
OTHER_COLORS = ["#1f77b4", "#9467bd", "#d62728", "#8c564b", "#e377c2",
                "#17becf", "#bcbd22", "#7f7f7f"]

# Text patterns used to spot numbered stage names:
#   _WEEK          "week8", "week_8", "Week 8" (any case) -> placed at that number
#   _NAMED_NUMBER  any word + number: "day3", "Session_2" -> put in number order
_WEEK = re.compile(r"^\s*week[\s_]*(\d+)\s*$", re.I)
_NAMED_NUMBER = re.compile(r"^\s*([A-Za-z][A-Za-z _-]*?)[\s_-]*(\d+)\s*$")

# Optional (leave as None unless you need them):
stage_mapping = None   # rename/group stages, e.g. {'baseline': 'Baseline', 'ldopa': 'L-DOPA'}
special_stages = None  # numbered drug stages, e.g. {25: 'L-Dopa', 26: 'Saline'}
stage_names = None     # your own tick labels, e.g. {1: 'Restricted', 2: 'Open'}

In [ ]:
# The per-window cluster table for ONE mouse (read-only: we never write to it).
# Change the paths accordingly 
Cluster_detail_results = pd.read_csv(r'Y:\3darena_behavior\lc_mitop_021626\mitopark\021626_2_mitop\ClusterResults_8-25LDOPA\Combined_Results\Results\test1\Cluster_detail_results.csv')

# Where the graphs go. A "transition_line_graphs" folder is made inside it.
# Use a folder of your own -- not the raw-data folders on the NAS.
output_dir = r'C:\Users\gangliagurdian\Desktop\transition_test_notebook\021626_2_mitop'

# Shown in the graph titles (also tells the script MitoPark = green, lc = orange).
mouse_name = 'MitoPark 021626_2_mitop'

# Check the stage names in this file:
Cluster_detail_results['Folder_Name'].unique()

In [ ]:
def count_transitions(df, cluster_col, stage_col):
    """List every transition in the table, one row each (stage, source, target).

    Same rules as Dylan's build_transitions: each window is paired with the
    next one, and the pair only counts if both windows are in the same
    (non-blank) stage and the cluster actually changes.
    """
    # np.roll(..., -1) shifts the list up by one, so tgt[i] is window i+1's cluster.
    src = df[cluster_col].to_numpy()            # this window's cluster
    tgt = np.roll(src, -1)                      # the next window's cluster
    stage = df[stage_col].to_numpy()            # this window's stage
    next_stage = np.roll(stage, -1)             # the next window's stage

    # Start with every pair kept, then switch off the ones that aren't real transitions.
    keep = np.ones(len(df), dtype=bool)
    keep[-1] = False                            # last window: no real "next" window
    keep &= (stage == next_stage)               # pair crosses into another stage
    keep &= pd.notna(stage)                     # blank row = break in the recording
    keep &= pd.notna(src) & pd.notna(tgt)       # a blank cluster number
    keep &= (src != tgt)                        # same cluster = no transition

    return pd.DataFrame({"stage": stage[keep],
                         "source": src[keep].astype(int),
                         "target": tgt[keep].astype(int)})


def pick_columns(df, cluster_col=None, stage_col=None):
    """Work out which columns hold the cluster and the stage."""
    if cluster_col and stage_col:
        pairs = [(cluster_col, stage_col)]      # the user told us
    else:
        pairs = COLUMN_PAIRS                    # guess from the usual names
    for c, s in pairs:
        if c in df.columns and s in df.columns:
            return c, s
    raise ValueError(f"Couldn't find cluster/stage columns {pairs} in the table. "
                     f"Columns available: {list(df.columns)}. Pass cluster_col= and "
                     "stage_col=.")

In [ ]:
def is_number(v):
    """True if the stage is just a number, like Week_Number 8 or 25.0."""
    try:
        float(v)
        return not isinstance(v, bool)
    except (TypeError, ValueError):
        return False


def stage_list(df, stage_col, order=None):
    """The stages in the order they first appear (or the order you give)."""
    seen = list(dict.fromkeys(df[stage_col].dropna()))
    if order is None:
        return seen
    # A typo in a given order would silently drop a stage, so check first.
    missing = [s for s in order if s not in seen]
    if missing:
        raise ValueError(f"stage_order names not found in the data: {missing}. "
                         f"Stages in the table: {seen}")
    return list(order)


def pretty(stage, stage_names=None, prefix="Week"):
    """The label under the x-axis: your own name > 8 -> 'Week 8' >
    'week25_LDOPA' -> 'Week 25 (LDOPA)' > underscores turned into spaces."""
    if stage_names and stage in stage_names:
        return str(stage_names[stage])
    if is_number(stage):
        n = float(stage)
        return f"{prefix} {int(n) if n.is_integer() else n}".strip()
    s = str(stage)
    m = re.match(r"^\s*week[\s_]*(\d+)(.*)$", s, re.I)
    if m:
        rest = m.group(2).replace("_", " ").strip()
        return f"Week {int(m.group(1))}" + (f" ({rest})" if rest else "")
    return s.replace("_", " ")


def stage_number(stage):
    """The week number if the stage has one (8, 'week8' -> 8.0), else None."""
    if is_number(stage):
        return float(stage)
    m = _WEEK.match(str(stage))
    return float(m.group(1)) if m else None


def map_stage(value, stage_map):
    """Rename one stage with stage_mapping (first key found in the name wins)
    or with a function like get_week_number."""
    if not isinstance(stage_map, dict):
        return stage_map(value)
    if isinstance(value, str):
        low = value.lower()
        for key, new in stage_map.items():
            if str(key).lower() in low:
                return new
    return stage_map.get(value, value)

In [ ]:
def cohort_of_text(text):
    """'mitopark', 'control', or None from a name like '021626_2_mitop'.

    control  = a piece "lc", a mouse ID ending in lc ("1lc", "f4lc"), or "control"
    MitoPark = a piece containing "mitop", or "mp", or a mouse ID ending in mp ("1mp")
    Both or neither (e.g. "lc_mitop_021626", "Session 1") -> None.
    """
    pieces = [p for p in re.split(r"[^a-z0-9]+", str(text).lower()) if p]
    control = any(p in ("control", "controls") or re.fullmatch(r"(?:[a-z]?\d+)?lc", p)
                  for p in pieces)
    mitopark = any("mitop" in p or re.fullmatch(r"(?:[a-z]?\d+)?mp", p)
                   for p in pieces)
    if control and not mitopark:
        return "control"
    if mitopark and not control:
        return "mitopark"
    return None


def find_cohort(item, name):
    """A file's group: from the name you gave it, else from its folders
    (starting next to the file and working up). Returns (group, folder)."""
    if name is not None:
        group = cohort_of_text(name)
        if group:
            return group, None
    if not isinstance(item, pd.DataFrame):
        for folder in reversed(Path(item).parent.parts):
            group = cohort_of_text(folder)
            if group:
                return group, folder
    return None, None


def shades(base, n, spread=0.34):
    """n shades of one color, darkest first (copied from Dylan's _shades)."""
    hue, light, sat = colorsys.rgb_to_hls(*to_rgb(base))
    if n <= 1:
        levels = [light]
    else:
        lo = max(light - spread / 2, 0.18)
        hi = min(light + spread / 2, 0.78)
        levels = np.linspace(lo, hi, n)
    return [colorsys.hls_to_rgb(hue, lv, sat) for lv in levels]


def line_colors(names, groups):
    """One color per file: orange shades for controls, green shades for
    MitoPark (sorted by name, like Dylan's cohort_colors), OTHER_COLORS for the rest."""
    out = [None] * len(names)
    for group, base in (("control", UTD_ORANGE), ("mitopark", UTD_GREEN)):
        members = sorted((str(names[i]), i) for i in range(len(names)) if groups[i] == group)
        for (_, i), col in zip(members, shades(base, len(members))):
            out[i] = col
    others = [i for i in range(len(names)) if out[i] is None]
    for k, i in enumerate(others):
        out[i] = OTHER_COLORS[k % len(OTHER_COLORS)]
    return out

In [ ]:
def load_session(data, name, color, cluster_col, stage_col, stage_map,
                 stage_order, separate_keywords, special_stages):
    """Read one table/file and work out everything needed to plot it."""
    df = data if isinstance(data, pd.DataFrame) else pd.read_csv(data)
    ccol, scol = pick_columns(df, cluster_col, stage_col)
    df = df.reset_index(drop=True)

    # Optionally rename/group stages first (stage_mapping); blank rows stay blank.
    if stage_map is not None:
        df = df.assign(**{scol: df[scol].map(
            lambda v: map_stage(v, stage_map) if pd.notna(v) else v)})
        if isinstance(stage_map, dict) and stage_order is None:
            wanted = list(dict.fromkeys(stage_map.values()))
            seen = list(dict.fromkeys(df[scol].dropna()))
            stage_order = [s for s in wanted if s in seen] + \
                          [s for s in seen if s not in wanted]

    trans = count_transitions(df, ccol, scol)
    if trans.empty:
        raise ValueError(f"{name}: no transitions found (check the stage column).")

    # Only stages that actually have transitions.
    present = set(trans["stage"])
    stages = [s for s in stage_list(df, scol, stage_order) if s in present]

    # Counts table: one row per (source, target), one column per stage.
    counts = (trans.groupby(["source", "target", "stage"]).size()
              .unstack("stage", fill_value=0).reindex(columns=stages, fill_value=0))
    # % of each stage's transitions (each column adds up to 100).
    pct = 100 * counts / counts.sum(axis=0)

    # Side (treatment) stages vs the main timeline.
    special = special_stages or {}
    side = [s for s in stages if s in special
            or (isinstance(s, str) and any(k in s.lower() for k in separate_keywords))]
    main = [s for s in stages if s not in side]
    if not main:
        main, side = side, []

    # Put stages in time order (files are often saved alphabetically).
    if all(stage_number(s) is not None for s in main):
        main = sorted(main, key=stage_number)
    elif stage_order is None and any(stage_number(s) is not None for s in main):
        # Weeks mixed with named stages: "pre..." first, then "base...",
        # then the weeks in number order, then any other named stages.
        weeks = sorted([s for s in main if stage_number(s) is not None], key=stage_number)
        named = [s for s in main if stage_number(s) is None]
        pre = [s for s in named if re.match(r"\s*pre", str(s), re.I)]
        base = [s for s in named if re.match(r"\s*base", str(s), re.I)]
        after = [s for s in named if s not in pre and s not in base]
        main = pre + base + weeks + after
    elif stage_order is None:
        # One shared word + number (day1 ... day10): put in number order.
        named = [_NAMED_NUMBER.match(str(s)) for s in main]
        if all(named) and len({m.group(1).lower() for m in named}) == 1:
            main = sorted(main, key=lambda s: int(_NAMED_NUMBER.match(str(s)).group(2)))
    stages = main + side

    return {"name": name, "color": color,
            "counts": counts[stages], "pct": pct[stages],
            "stages": stages, "main": main, "side": side,
            "total_main": counts[main].sum(axis=1)}

In [ ]:
def shared_axis(sessions, label):
    """Where each stage of each file sits on the x-axis (shared by all graphs)."""
    numbered = all(stage_number(s) is not None for ses in sessions for s in ses["main"])
    xpos = []
    if numbered:
        # x = week number; every week in range gets a tick (gaps show missing weeks).
        for ses in sessions:
            xpos.append({s: stage_number(s) for s in ses["main"]})
        lo = min(min(p.values()) for p in xpos)
        hi = max(max(p.values()) for p in xpos)
        ticks = [float(v) for v in range(int(np.floor(lo)), int(np.ceil(hi)) + 1)]
        stage_at = {x: s for ses, p in zip(sessions, xpos) for s, x in p.items()}
        labels = [label(stage_at[t]) if t in stage_at else label(int(t)) for t in ticks]
    else:
        # Named stages: lined up by position; different names -> "Baseline / Pre".
        n = max(len(ses["main"]) for ses in sessions)
        for ses in sessions:
            xpos.append({s: float(i) for i, s in enumerate(ses["main"])})
        ticks = [float(i) for i in range(n)]
        labels = []
        for i in range(n):
            names = list(dict.fromkeys(label(ses["main"][i]) for ses in sessions
                                       if i < len(ses["main"])))
            labels.append(" / ".join(names))

    # Side stages (drug sessions): one slot per label after the last tick.
    side_labels = list(dict.fromkeys(label(s) for ses in sessions for s in ses["side"]))
    slot = {lab: ticks[-1] + i for i, lab in enumerate(side_labels, start=1)}
    for p, ses in zip(xpos, sessions):
        for s in ses["side"]:
            p[s] = slot[label(s)]
    return xpos, ticks + list(slot.values()), labels + side_labels, numbered

In [ ]:
def plot_transition(src, tgt, rank, total, sessions, axis, title_label, label):
    """Draw the graph for one transition (src -> tgt) and return the figure."""
    xpos, ticks, tick_labels, numbered = axis
    fig, ax = plt.subplots(figsize=(8, 5))
    ymax = 0.0

    for ses, x in zip(sessions, xpos):
        # This transition's % per stage (0 everywhere if it never happens).
        row = (ses["pct"].loc[(src, tgt)] if (src, tgt) in ses["pct"].index
               else pd.Series(0.0, index=ses["stages"]))
        col = ses["color"]

        # Main timeline: solid line; NaN leaves a gap for unrecorded weeks.
        if numbered:
            lo = min(x[s] for s in ses["main"])
            hi = max(x[s] for s in ses["main"])
            xs = np.array([t for t in ticks if lo <= t <= hi])
            by_x = {x[s]: row[s] for s in ses["main"]}
            ys = np.array([by_x.get(v, np.nan) for v in xs], dtype=float)
        else:
            xs = np.array([x[s] for s in ses["main"]])
            ys = np.array([row[s] for s in ses["main"]], dtype=float)
        ax.plot(xs, ys, color=col, lw=2.5, marker="o", ms=5, label=ses["name"])

        # Drug sessions: dotted line from the last week to diamond markers.
        if ses["side"]:
            pts = sorted((x[s], row[s]) for s in ses["side"])
            xa = [x[ses["main"][-1]]] + [p[0] for p in pts]
            ya = [row[ses["main"][-1]]] + [p[1] for p in pts]
            ax.plot(xa, ya, color=col, lw=1.5, ls=":")
            ax.plot(xa[1:], ya[1:], color=col, ls="none", marker="D", ms=6)

        ymax = max(ymax, float(np.nanmax(row.to_numpy(dtype=float))))

    # Tilt the labels when there are many stages.
    ax.set_xticks(ticks)
    many = len(ticks) > 6
    ax.set_xticklabels(tick_labels, rotation=60 if many else 0,
                       ha="right" if many else "center", fontsize=8 if many else 10)
    if len(ticks) <= 3:
        ax.set_xlim(min(ticks) - 0.5, max(ticks) + 0.5)

    ax.set_ylim(0, max(ymax * 1.15, 0.01))
    ax.set_ylabel("% of that stage's transitions")

    # Title (one file) or title + legend with counts (several files).
    if len(sessions) == 1:
        ses = sessions[0]
        n = int(ses["total_main"].get((src, tgt), 0))
        ax.set_title(f"{title_label}: transition {src} → {tgt}\n"
                     f"rank {rank} of {total} · {n} times over "
                     f"{label(ses['main'][0])} – {label(ses['main'][-1])}",
                     fontsize=11)
    else:
        ax.set_title(f"{title_label}: transition {src} → {tgt}\n"
                     f"rank {rank} of {total} (all files combined)", fontsize=11)
        handles, names = ax.get_legend_handles_labels()
        names = [f"{nm} ({int(ses['total_main'].get((src, tgt), 0))} times)"
                 for nm, ses in zip(names, sessions)]
        ax.legend(handles, names, fontsize=9, frameon=False)

    # Lab style: no top/right box lines.
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    fig.tight_layout()
    return fig

In [ ]:
def make_transition_line_graphs(data, out_dir=None, top_n=TOP_N, stage_order=None,
                                label=None, color=None, names=None, colors=None,
                                cluster_col=None, stage_col=None, stage_map=None,
                                special_stages=None, stage_names=None,
                                stage_prefix="Week", formats=("jpeg", "svg"),
                                dpi=130, show=False,
                                separate_keywords=SEPARATE_KEYWORDS):
    """Count the transitions and make a graph for each of the top_n."""
    # Never write into a folder that already has files (no overwriting).
    if out_dir is not None:
        out_dir = Path(out_dir)
        if out_dir.exists() and any(out_dir.iterdir()):
            print(f"{out_dir} already has files in it -- skipped so nothing is "
                  "overwritten. Move or rename it to re-make these graphs.")
            return None

    # No Pillow -> save .png instead of .jpeg.
    formats = list(formats)
    if any(f.lower() in ("jpeg", "jpg") for f in formats):
        try:
            import PIL  # noqa: F401
        except ImportError:
            print("Pillow isn't installed, so .jpeg can't be saved -- saving .png "
                  "instead (pip install pillow to get .jpeg).")
            formats = list(dict.fromkeys("png" if f.lower() in ("jpeg", "jpg") else f
                                         for f in formats))

    items = list(data) if isinstance(data, (list, tuple)) else [data]

    # Names, groups (MitoPark / control / neither) and default legend names.
    if not names and len(items) == 1 and label:
        names = [label]
    given = list(names) if names else [None] * len(items)
    if len(given) != len(items):
        raise ValueError(f"Got {len(items)} tables/files but {len(given)} names.")
    found = [find_cohort(it, nm) for it, nm in zip(items, given)]
    groups = [grp for grp, _ in found]

    def default_name(item, i):
        if isinstance(item, pd.DataFrame):
            return f"file {i + 1}"
        return found[i][1] or Path(item).parent.name
    names = [nm if nm is not None else default_name(it, i)
             for i, (it, nm) in enumerate(zip(items, given))]

    # Colors: yours if given, else by group (green / orange / others).
    if colors:
        cols = [list(colors)[i % len(colors)] for i in range(len(items))]
    elif len(items) == 1 and color:
        cols = [color]
    else:
        cols = line_colors(names, groups)

    sessions = [load_session(it, nm, cols[i], cluster_col, stage_col,
                             stage_map, stage_order, separate_keywords, special_stages)
                for i, (it, nm) in enumerate(zip(items, names))]
    if len(items) > 1:
        print("Comparing files: make sure they were clustered TOGETHER, so the "
              "same cluster number is the same behavior in each.")

    labels_given = dict(special_stages or {})
    labels_given.update(stage_names or {})
    def stage_label(s):
        return pretty(s, labels_given, stage_prefix)

    title_label = label or (names[0] if len(items) == 1 else " vs ".join(names))

    # Rank by count over the main stages (all files added up); ties broken
    # by source then target so the order is the same on every computer.
    totals = pd.concat([s["total_main"] for s in sessions], axis=1).fillna(0)
    combined = totals.sum(axis=1)
    ranked = combined.rename("n").reset_index()
    ranked = ranked.sort_values(["n", "source", "target"], ascending=[False, True, True])
    order = pd.MultiIndex.from_frame(ranked[["source", "target"]])
    top = order[:top_n]

    axis = shared_axis(sessions, stage_label)
    if out_dir is not None:
        out_dir.mkdir(parents=True, exist_ok=True)

    # One graph per top transition, e.g. transition_01_045_to_012.
    rows = []
    for rank, (src, tgt) in enumerate(top, start=1):
        fname = f"transition_{rank:02d}_{int(src):03d}_to_{int(tgt):03d}"
        fig = plot_transition(src, tgt, rank, len(order), sessions, axis,
                              title_label, stage_label)
        if out_dir is not None:
            for ext in formats:
                stem = ("_" + fname) if ext == "svg" else fname
                fig.savefig(out_dir / f"{stem}.{ext}",
                            dpi=dpi if ext != "svg" else None)
        if show:
            plt.show()
        else:
            plt.close(fig)

        row = {"rank": rank, "source": int(src), "target": int(tgt),
               "count_main_stages": int(combined[(src, tgt)])}
        if len(sessions) > 1:
            for ses in sessions:
                row[f"count_{ses['name']}"] = int(ses["total_main"].get((src, tgt), 0))
        row["file"] = f"{fname}.{formats[0]}" if out_dir is not None else ""
        rows.append(row)
    index = pd.DataFrame(rows)

    # Every transition's % in every stage.
    parts = []
    for ses in sessions:
        p = ses["pct"].reindex(order).fillna(0).round(4)
        prefix = f"{ses['name']}: " if len(sessions) > 1 else ""
        p.columns = [prefix + stage_label(s) for s in p.columns]
        parts.append(p)
    table = pd.concat(parts, axis=1)
    table.insert(0, "count_main_stages", combined.loc[order].astype(int))
    table.index.names = ["source", "target"]
    if out_dir is not None:
        index.to_csv(out_dir / "transition_index.csv", index=False)
        table.to_csv(out_dir / "all_transitions_by_stage.csv")

    # Summary, e.g. "021626_2_mitop [MitoPark, green]: 7086 distinct transitions, ..."
    group_text = {"mitopark": "MitoPark, green", "control": "control, orange",
                  None: "not MitoPark/control"}
    for ses, grp in zip(sessions, groups):
        how = "your color" if (colors or (len(items) == 1 and color)) else group_text[grp]
        print(f"{ses['name']} [{how}]: {len(ses['counts'])} distinct transitions, "
              f"{int(ses['counts'].to_numpy().sum())} in total; stages: "
              + ", ".join(stage_label(s) for s in ses["stages"])
              + (f"  (side: {', '.join(stage_label(s) for s in ses['side'])})"
                 if ses["side"] else ""))
    print(f"  made top {len(top)} graphs"
          + (f" + transition_index.csv + all_transitions_by_stage.csv\n  into {out_dir}"
             if out_dir is not None else " (not saved)"))
    return index


def create_transition_line_charts(df, save=False, output_dir=None, mouse_name=None,
                                  top_n=TOP_N, **options):
    """Notebook version: save=False shows the graphs; save=True saves .png + .svg
    (300 dpi) into <output_dir>/transition_line_graphs."""
    out = Path(output_dir) / "transition_line_graphs" if (save and output_dir) else None
    options.setdefault("formats", ("png", "svg"))
    options.setdefault("dpi", 300)
    return make_transition_line_graphs(df, out, top_n=top_n, label=mouse_name,
                                       show=not save, **options)

In [ ]:
create_transition_line_charts(Cluster_detail_results, save=False, mouse_name=mouse_name,
                              top_n=TOP_N, stage_map=stage_mapping,
                              special_stages=special_stages, stage_names=stage_names)

In [ ]:
create_transition_line_charts(Cluster_detail_results, save=True, output_dir=output_dir,
                              mouse_name=mouse_name, top_n=TOP_N, stage_map=stage_mapping,
                              special_stages=special_stages, stage_names=stage_names)

In [ ]:
# mouse_1 = pd.read_csv(r'...\Cluster_detail_results.csv')
# mouse_2 = pd.read_csv(r'...\Cluster_detail_results.csv')
# mouse_3 = pd.read_csv(r'...\Cluster_detail_results.csv')
#
# create_transition_line_charts([mouse_1, mouse_2, mouse_3], save=False,
#                               names=['MitoPark 021626_2_mitop', 'control 021626_1_lc', 'MitoPark 042025_1mp'],
#                               top_n=TOP_N)